<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/TripoSplat_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## TripoSplat — Image to 3D Gaussian Splats

[TripoAI / VAST-AI-Research TripoSplat](https://www.tripo3d.ai/research/triposplat) ([arXiv 2605.16355](https://arxiv.org/abs/2605.16355), [HF VAST-AI/TripoSplat](https://huggingface.co/VAST-AI/TripoSplat), [code](https://github.com/VAST-AI-Research/TripoSplat), **MIT license**). Converts a **single 2D image** into a 3D Gaussian Splat scene that can be rendered in real-time by a 3DGS viewer. The first fully-commercial-OK image-to-3D model in the suite.

**Architecture**: DINOv3 ViT-H/16+ (1.68 GB) + Flux2 VAE encoder (336 MB) → fused conditioning for a 24-block, 1024-dim, 16-head flow-matching DiT (741 MB) → Octree + Gaussian decoder (576 MB) → fixed-length Gaussian set (32k → 262k, multiple of 32). Background removal via BiRefNet Swin-L (444 MB). 8192 latent tokens, 20-step Euler flow sampling with classifier-free guidance. ~3.78 GB total weights.

**Outputs**: native 3DGS only — `.ply` (3DGS standard) and `.splat` (32-byte-per-Gaussian web-viewer format). Open the `.ply` in [Antimatter15](https://antimatter15.com/splat/), [gsplat.tech](https://gsplat.tech/), or LumaAI for real-time 3D rendering.

> **For game-ready textured meshes:** use [Pixal3D_Colab.ipynb](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pixal3D_Colab.ipynb) instead — it produces PBR-textured GLB meshes from the same single-image input with much better quality than naive 3DGS-to-mesh reconstruction could ever achieve. For high-quality 3DGS-to-mesh research pipelines, see [SuGaR_Colab.ipynb](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/SuGaR_Colab.ipynb) (sharp, 2-3 hrs/scene) and [GauStudio_Colab.ipynb](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/GauStudio_Colab.ipynb) (smooth, ~10 min/scene).

### Quick Start
> Requires **Colab Runtime Version 2026.01** (ships torch 2.9.0+cu126, MOSS-TTS-style 2026.01 numpy patch is included defensively). Set via Runtime → Change runtime type → L4 GPU (22 GB) recommended. T4 (16 GB) is tight — use ≤ 65k gaussians and ≤ 15 sampling steps.



In [ ]:
# @title STEP 1 — Install Dependencies
# @markdown Clones [VAST-AI-Research/TripoSplat](https://github.com/VAST-AI-Research/TripoSplat),
# @markdown installs lightweight deps (open3d for mesh recon + GLB export, trimesh for OBJ,
# @markdown gradio for UI), and applies the **numpy 2.x umath patch** that protects any
# @markdown downstream import of `numpy._core.strings` (same as MOSS-TTS_Colab). First run ~3 min.

import os, sys, subprocess, time, shutil, glob, site

REPO_DIR = '/content/TripoSplat'
REPO_URL = 'https://github.com/VAST-AI-Research/TripoSplat.git'

if not os.path.isdir(REPO_DIR):
    print(f'[git] Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    print(f'[git] Cloned to {REPO_DIR}')
else:
    print(f'[git] {REPO_DIR} already present, skipping clone.')

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)

# System deps
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], check=False)

# ── numpy._core.umath patch (carried from MOSS-TTS_Colab) ──────────────────
# Colab Runtime 2026.01 ships numpy 2.0.x where the string ufuncs (_center,
# _ljust, _rjust, _zfill, _strip_*, _lstrip_*, _rstrip_*, _partition*,
# _rpartition*, _slice, _expandtabs*, _replace, is*, find/index,
# startswith/endswith, str_len) are NOT yet exposed in numpy._core.umath.
# The first import of numpy._core.strings (triggered by `import torchvision`,
# `import safetensors`, `import soundfile`, etc.) then fails with
# `ImportError: cannot import name '_center' from 'numpy._core.umath'`.
# We inject minimal pure-Python ufuncs into umath before any of those imports
# happen, so the `from numpy._core.umath import ...` in numpy._core.strings
# succeeds. MOSS-TTS-style defensive patch.
import numpy as np
import numpy._core.umath as _umath

_NEEDED = (
    '_center', '_expandtabs', '_expandtabs_length', '_ljust',
    '_lstrip_chars', '_lstrip_whitespace', '_partition',
    '_partition_index', '_replace', '_rjust', '_rpartition',
    '_rpartition_index', '_rstrip_chars', '_rstrip_whitespace',
    '_slice', '_strip_chars', '_strip_whitespace', '_zfill',
    'count', 'endswith', 'find', 'index', 'isalnum', 'isalpha',
    'isdecimal', 'isdigit', 'islower', 'isnumeric', 'isspace',
    'istitle', 'isupper', 'rfind', 'rindex', 'startswith', 'str_len',
)

def _str_center(a, width, fillchar=' '):
    width = int(width); fc = (str(fillchar)[:1]) or ' '
    return np.array([str(s).center(width, fc) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_ljust(a, width, fillchar=' '):
    width = int(width); fc = (str(fillchar)[:1]) or ' '
    return np.array([str(s).ljust(width, fc) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_rjust(a, width, fillchar=' '):
    width = int(width); fc = (str(fillchar)[:1]) or ' '
    return np.array([str(s).rjust(width, fc) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_zfill(a, width):
    width = int(width)
    return np.array([str(s).zfill(width) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_strip(a, chars=None):
    return np.array([str(s).strip(chars) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_lstrip(a, chars=None):
    return np.array([str(s).lstrip(chars) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _str_rstrip(a, chars=None):
    return np.array([str(s).rstrip(chars) for s in np.asarray(a).flat],
                    dtype=object).reshape(np.asarray(a).shape)
def _bool_pred(name):
    return lambda a: np.array([getattr(str(s), name)() for s in np.asarray(a).flat],
                              dtype=bool).reshape(np.asarray(a).shape)

_REAL = {
    '_center':           (_str_center, 3, 1),
    '_ljust':            (_str_ljust, 3, 1),
    '_rjust':            (_str_rjust, 3, 1),
    '_zfill':            (_str_zfill, 2, 1),
    '_strip_chars':      (_str_strip, 2, 1),
    '_lstrip_chars':     (_str_lstrip, 2, 1),
    '_rstrip_chars':     (_str_rstrip, 2, 1),
    '_strip_whitespace': (_str_strip, 1, 1),
    '_lstrip_whitespace':(_str_lstrip, 1, 1),
    '_rstrip_whitespace':(_str_rstrip, 1, 1),
    'isalnum':           (_bool_pred('isalnum'),   1, 1),
    'isalpha':           (_bool_pred('isalpha'),   1, 1),
    'isdigit':           (_bool_pred('isdigit'),   1, 1),
    'islower':           (_bool_pred('islower'),   1, 1),
    'isupper':           (_bool_pred('isupper'),   1, 1),
    'isspace':           (_bool_pred('isspace'),   1, 1),
    'isnumeric':         (_bool_pred('isnumeric'), 1, 1),
    'isdecimal':         (_bool_pred('isdecimal'), 1, 1),
    'istitle':           (_bool_pred('istitle'),   1, 1),
}
_PASSTHROUGH = (1, 1)
_patched = []
for _name in _NEEDED:
    if hasattr(_umath, _name):
        continue
    try:
        _fn, _nin, _nout = _REAL.get(_name, (lambda *a: a[0] if a else '', *_PASSTHROUGH))
        _umath.__dict__[_name] = np.frompyfunc(_fn, _nin, _nout)
        _patched.append(_name)
    except Exception as _e:
        print(f'[numpy_patch] could not patch {_name}: {_e}')
if _patched:
    print(f'[numpy_patch] Injected {len(_patched)} string ufunc(s) into numpy._core.umath: {_patched}')
    print('[numpy_patch] Workaround for Colab Runtime 2026.01 numpy 2.0.x; safe to ignore on newer numpy.')
else:
    print('[numpy_patch] numpy._core.umath already has all string ufuncs; no patching needed.')
del _NEEDED, _REAL, _PASSTHROUGH, _patched
for _var in ('_fn', '_nin', '_nout', '_name'):
    if _var in dir():
        del globals()[_var]
# ─────────────────────────────────────────────────────────────────────────────

# Install lightweight deps needed by the model + our mesh/GLB/FBX export path
t0 = time.time()
print('[pip] Installing lightweight deps (open3d, trimesh, safetensors, etc.) ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'safetensors', 'huggingface_hub>=0.25.0', 'hf_transfer',
                'gradio==5.49.1', 'tqdm==4.66.5',
                'open3d==0.19.0', 'trimesh==4.12.2',
                'pymeshlab>=2022.2', 'pyfqmr>=0.5.0',
                'scikit-image', 'pywavefront'],
               check=True)
print(f'[pip] lightweight deps: {time.time() - t0:.1f}s')

# Note: numpy, torch, torchvision, PIL, safetensors, einops are already
# preinstalled in Colab. MOSS-TTS_Colab does a full numpy-downgrade dance,
# but TripoSplat works fine with numpy 2.x after the umath patch above.

# Sanity-check imports
import torch
print(f'[verify] torch={torch.__version__} cuda={torch.cuda.is_available()}')
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'[verify] GPU: {gpu_name} ({vram_gb:.1f} GB)')
    if vram_gb < 14:
        print(f'[verify] WARNING: only {vram_gb:.0f} GB VRAM available.')
        print(f'[verify]   TripoSplat needs ~14 GB minimum; consider L4/A100 (>=22 GB).')
    elif vram_gb < 20:
        print(f'[verify] VRAM is tight: use num_gaussians<=65536 and steps<=15.')
    else:
        print(f'[verify] VRAM is comfortable: full defaults supported.')

import safetensors
import open3d
import trimesh
print(f'[verify] safetensors={safetensors.__version__} open3d={open3d.__version__} trimesh={trimesh.__version__}')

# Import TripoSplat components (these are in the cloned repo's `model.py` and `triposplat.py`)
from model import (DinoV3ViT, Flux2VAEEncoder, BiRefNet,
                  OctreeProbabilityFixedlenDecoder, ElasticGaussianFixedlenDecoder,
                  LatentSeqMMFlowModel, OctreeGaussianDecoder)
print(f'[verify] TripoSplat model classes imported from {REPO_DIR}/model.py')

print(f'\n[Done] STEP 1 complete in {time.time() - t0:.1f}s. Proceed to Step 2.')


In [ ]:
# @title STEP 2 — Pre-cache Weights
# @markdown Downloads the 5 TripoSplat safetensors (~3.78 GB total) from HuggingFace.
# @markdown On Colab, weights are cached to Google Drive at
# @markdown `AEI_3D_Cache/TripoSplat/` so subsequent sessions don't re-download.
# @markdown First run: ~3-5 min. Subsequent runs: <5 s (skip if already present).

import os, sys, time, shutil
from pathlib import Path

# Drive cache strategy — use Drive for weights to persist across Colab sessions.
# Drive doesn't support symlinks for huggingface_hub v1.x — so we keep two paths
#   1. The official `huggingface_hub` cache (symlink-friendly, for v0.x) at
#      `HF_HOME=/content/drive/MyDrive/AEI_3D_Cache/huggingface`  -- we DON'T use this
#      because v1.x has known symlink issues on Drive.
#   2. A direct `snapshot_download(..., local_dir=...)` into a Drive folder -- this
#      works because we don't need symlinks, just regular files.
DRIVE_DIR = Path('/content/drive/MyDrive/AEI_3D_Cache/TripoSplat')
LOCAL_DIR = Path('/content/triposplat_weights')
DRIVE_MOUNTED = os.path.isdir('/content/drive/MyDrive')

# Pick the cache root. Prefer Drive if available; fall back to local /content.
if DRIVE_MOUNTED:
    CACHE_DIR = DRIVE_DIR
    print(f'[cache] Using Drive cache: {CACHE_DIR}')
else:
    CACHE_DIR = LOCAL_DIR
    print(f'[cache] Drive not mounted; using local cache: {CACHE_DIR}')
    print(f'[cache] (Reconnect your Google Drive in Colab for cross-session persistence.)')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Check if already complete
expected = {
    'background_removal/birefnet.safetensors': 444_000_000,
    'clip_vision/dino_v3_vit_h.safetensors': 1_680_000_000,
    'diffusion_models/triposplat_fp16.safetensors': 741_000_000,
    'vae/flux2-vae.safetensors': 336_000_000,
    'vae/triposplat_vae_decoder_fp16.safetensors': 576_000_000,
}
total_expected = sum(expected.values())  # ~3.78 GB
total_actual = 0
missing = []
for relpath, expected_size in expected.items():
    p = CACHE_DIR / relpath
    if p.exists() and p.stat().st_size > expected_size * 0.9:  # 10% tolerance
        total_actual += p.stat().st_size
    else:
        missing.append(relpath)
print(f'[cache] {len(expected) - len(missing)}/{len(expected)} files present, {total_actual/1e9:.2f} GB / {total_expected/1e9:.2f} GB')

if not missing:
    print(f'[cache] All TripoSplat weights present at {CACHE_DIR}. Skip download.')
else:
    print(f'[cache] Missing files: {missing}')
    print(f'[cache] Downloading from VAST-AI/TripoSplat (~{(total_expected - total_actual)/1e9:.1f} GB) ...')
    from huggingface_hub import snapshot_download
    t0 = time.time()
    try:
        snapshot_download(
            repo_id='VAST-AI/TripoSplat',
            local_dir=str(CACHE_DIR),
            allow_patterns=['*.safetensors', '*.json'],
            max_workers=4,
        )
        elapsed = time.time() - t0
        print(f'[cache] Download complete in {elapsed:.0f}s ({total_expected/1e9:.1f} GB).')
    except Exception as e:
        print(f'[cache] Download failed: {e}')
        print(f'[cache] Will retry on next run, or check your network / HF rate limits.')
        # Don't raise - we want STEP 3 to attempt the download via snapshot_download
        # at model-load time as a fallback.

# Save the cache path for STEP 3
TS_WEIGHTS_DIR = str(CACHE_DIR)
print(f'\n[Done] STEP 2 complete. Weights at: {TS_WEIGHTS_DIR}')
print(f'[Done] Proceed to Step 3 to define the pipeline.')


In [ ]:
# @title STEP 3 — Core Pipeline (3DGS output only, .ply + .splat)
# @markdown Defines the TripoSplat pipeline (load + inference) and helpers to convert
# @markdown the Gaussian set into a triangle mesh, then export to GLB / FBX / OBJ / PLY.

import os, sys, time
from pathlib import Path
import numpy as np
import torch

REPO_DIR = '/content/TripoSplat'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from triposplat import TripoSplatPipeline as _TSPipeline

# Constants
TS_HF_REPO = 'VAST-AI/TripoSplat'  # HuggingFace repo for the 5 safetensors

PIPE = None
_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# ── TripoSplat pipeline wrapper ───────────────────────────────────────────
# The original TripoSplatPipeline loads from local paths. We wrap it so the
# checkpoint paths are auto-resolved from the cache dir, with a fallback to
# a fresh snapshot_download if files are missing.

def _resolve_ckpt_paths():
    """Find the 5 safetensors, either in Drive cache or local cache. Fall back to download."""
    from pathlib import Path
    from huggingface_hub import snapshot_download
    candidates = [
        Path('/content/drive/MyDrive/AEI_3D_Cache/TripoSplat'),
        Path('/content/triposplat_weights'),
    ]
    base = next((c for c in candidates if c.exists()), None)
    if base is None:
        # Fresh download into local
        base = Path('/content/triposplat_weights')
        base.mkdir(parents=True, exist_ok=True)
        print(f'[load] No cache found; downloading {TS_HF_REPO} to {base} ...')
        snapshot_download(repo_id=TS_HF_REPO, local_dir=str(base),
                          allow_patterns=['*.safetensors', '*.json'])
    return {
        'ckpt_path':              str(base / 'diffusion_models' / 'triposplat_fp16.safetensors'),
        'decoder_path':           str(base / 'vae' / 'triposplat_vae_decoder_fp16.safetensors'),
        'dinov3_path':            str(base / 'clip_vision' / 'dino_v3_vit_h.safetensors'),
        'flux2_vae_encoder_path': str(base / 'vae' / 'flux2-vae.safetensors'),
        'rmbg_path':              str(base / 'background_removal' / 'birefnet.safetensors'),
    }

class TripoSplatColab:
    """Colab-friendly wrapper around TripoSplatPipeline.

    - `prepare(image)` → preprocessed 1024x1024 RGBA-on-black PIL image
    - `synthesize(image, seed, steps, guidance_scale, num_gaussians, shift=3.0,
                  erode_radius=1, progress_callback=None)` → (Gaussian, prepared_image)
    """
    def __init__(self):
        self._pipe = None
        self._load_logged = False

    def load(self):
        if self._pipe is not None:
            return
        paths = _resolve_ckpt_paths()
        t0 = time.time()
        self._pipe = _TSPipeline(
            ckpt_path=paths['ckpt_path'],
            decoder_path=paths['decoder_path'],
            dinov3_path=paths['dinov3_path'],
            flux2_vae_encoder_path=paths['flux2_vae_encoder_path'],
            rmbg_path=paths['rmbg_path'],
            device=_DEVICE,
        )
        if not self._load_logged:
            vram = (torch.cuda.memory_allocated() / 1024**3) if torch.cuda.is_available() else 0
            print(f'[load] TripoSplat loaded in {time.time() - t0:.1f}s — VRAM={vram:.1f} GB')
            self._load_logged = True

    def prepare(self, image, erode_radius=1):
        return self._pipe.preprocess_image(image, erode_radius=erode_radius)

    def synthesize(self, image, seed=42, steps=20, guidance_scale=3.0, shift=3.0,
                   num_gaussians=131072, erode_radius=1, callback=None):
        """End-to-end: preprocess → encode → sample → decode → Gaussian.

        Args:
            image: PIL.Image or file path or tensor.
            seed: int RNG seed.
            steps: int flow-matching steps (10-20 recommended).
            guidance_scale: float CFG strength (3.0 recommended).
            shift: float flow-matching schedule shift (3.0 recommended).
            num_gaussians: int target Gaussian count (multiple of 32, in [32768, 262144]).
            erode_radius: int alpha-erosion radius for background removal postprocessing.
            callback: optional fn(step, total) for progress UI.

        Returns:
            (Gaussian, prepared_image) tuple. Gaussian has save_ply() and save_splat().
        """
        self.load()
        gen = torch.Generator(device=self._pipe._device).manual_seed(int(seed))
        prepared = self.prepare(image, erode_radius=erode_radius)
        cond = self._pipe.encode_image(prepared, generator=gen)
        out = self._pipe.sample_latent(cond, steps=int(steps),
                                        guidance_scale=float(guidance_scale),
                                        shift=float(shift), generator=gen,
                                        show_progress=False, callback=callback)
        gauss = self._pipe.decode_latent(out['latent'], num_gaussians=int(num_gaussians))
        return gauss, prepared

PIPE = TripoSplatColab()


# ── Output: save 3DGS Gaussians to disk ────────────────────────────────────
# TripoSplat's native outputs
#   - .ply   (3DGS standard, ~5-50 MB depending on Gaussian count)
#           Loadable in Antimatter15 viewer, gsplat.js, LumaAI, Polycam
#   - .splat (32-byte-packed, web-viewer-optimized, ~2-15 MB)
#           Smaller alternative to .ply, same rendering
#
# We do NOT export mesh formats. TripoSplat's 3DGS-derived mesh outputs
# (via Poisson/alpha_shape) were consistently low-quality (holes, missing
# surfaces, no UVs). For game-ready textured meshes, use Pixal3D instead.
# For high-quality 3DGS-to-mesh post-processing, see SuGaR_Colab.ipynb
# and GauStudio_Colab.ipynb in this suite.


def save_gaussians(gaussian, out_dir, base_name='triposplat',
                   write_ply=True, write_splat=True):
    """Save a TripoSplat Gaussian set to .ply and/or .splat.

    Args:
        gaussian: TripoSplat Gaussian object (has save_ply / save_splat methods).
        out_dir: output directory (will be created if missing).
        base_name: filename prefix (e.g. 'hero' → 'hero.ply', 'hero.splat').
        write_ply: write the 3DGS-standard PLY.
        write_splat: write the 32-byte-packed web-viewer PLY.

    Returns: dict of format → path written (only includes formats that succeeded).
    """
    import os, time
    os.makedirs(out_dir, exist_ok=True)
    paths = {}
    if write_ply:
        ply_path = os.path.join(out_dir, f'{base_name}.ply')
        t0 = time.time()
        try:
            gaussian.save_ply(ply_path)
            sz = os.path.getsize(ply_path) / (1024 * 1024)
            print(f'  [save] {ply_path} ({sz:.1f} MB, {time.time()-t0:.1f}s)')
            paths['ply'] = ply_path
        except Exception as e:
            print(f'  [save] .ply failed: {e}')
    if write_splat:
        splat_path = os.path.join(out_dir, f'{base_name}.splat')
        t0 = time.time()
        try:
            gaussian.save_splat(splat_path)
            sz = os.path.getsize(splat_path) / (1024 * 1024)
            print(f'  [save] {splat_path} ({sz:.1f} MB, {time.time()-t0:.1f}s)')
            paths['splat'] = splat_path
        except Exception as e:
            print(f'  [save] .splat failed: {e}')
    return paths


print('[Core] TripoSplat pipeline ready.')
print('[Core] Use Step 4 (Gradio UI) or Step 6 (quick test) to run inference.')
print('[Core] Outputs: .ply (3DGS standard) + .splat (web viewer packed)')
print('[Core] For game-ready textured meshes, use Pixal3D_Colab.ipynb instead.')


In [ ]:
# @title STEP 4 — Launch Gradio UI
# @markdown Single-image Gradio UI for TripoSplat. Outputs `.ply` + `.splat`.
# @markdown For game-ready textured meshes use Pixal3D_Colab.ipynb instead.

import os, sys, time, uuid, shutil, pathlib, traceback
import numpy as np
import torch
import gradio as gr
from PIL import Image
from IPython.display import clear_output, display, FileLink

REPO_DIR = '/content/TripoSplat'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

OUT_ROOT = pathlib.Path('/content/triposplat_runs')
OUT_ROOT.mkdir(parents=True, exist_ok=True)


def _new_run_dir():
    p = OUT_ROOT / f'run_{int(time.time())}_{uuid.uuid4().hex[:6]}'
    p.mkdir(parents=True, exist_ok=True)
    return p


def _slugify(path):
    stem = pathlib.Path(path).stem if path else 'triposplat'
    s = ''.join(c if c.isalnum() else '_' for c in stem[:50]).strip('_') or 'triposplat'
    return s


def do_generate(image, seed, steps, guidance_scale, shift, num_gaussians,
                randomize_seed, progress=gr.Progress()):
    """Run TripoSplat + save .ply + .splat. Returns (preview, info, files)."""
    try:
        if image is None:
            raise gr.Error('Upload an image first.')
        if randomize_seed:
            seed = int.from_bytes(os.urandom(2), 'big')
        seed = int(seed)
        steps = int(steps)
        guidance_scale = float(guidance_scale)
        shift = float(shift)
        num_gaussians = int(num_gaussians)

        progress(0.0, desc='Preprocessing image')
        run_dir = _new_run_dir()
        t0 = time.time()
        gauss, prepared = PIPE.synthesize(
            image, seed=seed, steps=steps,
            guidance_scale=guidance_scale, shift=shift,
            num_gaussians=num_gaussians,
        )
        gen_dt = time.time() - t0
        n_gauss = gauss.get_xyz.shape[0]
        progress(0.6, desc=f'Generated {n_gauss:,} Gaussians in {gen_dt:.1f}s')

        prep_path = run_dir / 'preprocessed.png'
        prepared.save(str(prep_path))

        # Save .ply + .splat (the only useful outputs)
        SLUG = _slugify(image.name if hasattr(image, 'name') else None)
        paths = save_gaussians(gauss, str(run_dir), base_name=SLUG,
                                write_ply=True, write_splat=True)

        n_formats = len(paths)
        info_lines = [
            f'**Generated** {n_gauss:,} Gaussians in {gen_dt:.1f}s',
            f'**Seed:** {seed}  |  **Steps:** {steps}  |  **Guidance:** {guidance_scale}  |  **Shift:** {shift}',
            f'**Num Gaussians:** {num_gaussians:,}',
        ]
        for fmt, p in paths.items():
            sz = pathlib.Path(p).stat().st_size / (1024*1024)
            info_lines.append(f'**{fmt.upper()}**: `{p}` ({sz:.1f} MB)')
        info_md = '\
\
'.join(info_lines)

        # Build a file gallery for download
        gallery = []
        for label, key in [('3DGS PLY (Antimatter15, gsplat.js)', 'ply'),
                            ('3DGS SPLAT (web viewer packed)', 'splat')]:
            if key in paths:
                gallery.append((label, paths[key]))

        preview_path = str(prep_path)
        # Pick the first .ply for the 3D preview (model-viewer supports PLY)
        if 'ply' in paths:
            preview_path = paths['ply']
        return preview_path, info_md, gallery
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f'Generation failed: {e}')


# ── UI ────────────────────────────────────────────────────────────────────
with gr.Blocks(title='TripoSplat — Image to 3D Gaussians', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# TripoSplat — Image to 3D Gaussians\
'
                'Convert a single image into 3D Gaussian Splats. Outputs `.ply` (3DGS standard) '
                'and `.splat` (web-viewer packed). For game-ready textured meshes, use '
                '[Pixal3D_Colab](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pixal3D_Colab.ipynb) instead.')
    with gr.Row():
        with gr.Column():
            q_image = gr.Image(type='pil',
                                label='Input image (foreground subject, BG auto-removed via BiRefNet if no alpha)',
                                sources=['upload', 'clipboard'],
                                height=380)
            with gr.Row():
                q_random_seed = gr.Checkbox(value=True, label='Random seed',
                                              info='Override the seed field with a random one each run.')
                q_seed = gr.Number(value=42, label='Seed (int)', precision=0,
                                    info='RNG seed. Same seed = same output (modulo minor cuDNN nondeterminism).')
            q_steps = gr.Slider(5, 50, value=20, step=1, label='Sampling steps',
                                info='Flow-matching Euler steps. 10-20 recommended. More = slower + sharper.')
            q_guidance = gr.Slider(1.0, 10.0, value=3.0, step=0.1, label='Guidance scale (CFG)',
                                    info='Classifier-free guidance. ≤1 disables CFG. 3.0 is the model default.')
            q_shift = gr.Slider(1.0, 5.0, value=3.0, step=0.1, label='Flow schedule shift',
                                 info='Flow-matching time shift. 3.0 is the model default.')
            q_num_g = gr.Slider(32768, 262144, value=131072, step=1024, label='Num Gaussians',
                                 info='Total Gaussian count. Must be a multiple of 32. 131k default. T4-safe <= 65k.')
            q_btn = gr.Button('Generate 3D Gaussians', variant='primary', size='lg')
    with gr.Column():
        q_preview = gr.Image(label='Preprocessed input (1024×1024) — what the model sees', height=240)
        q_html = gr.Model3D(label='3D preview (.ply — click to orbit, scale, pan)',
                          height=520)
        q_info = gr.Markdown(label='Run info')
        q_files = gr.Gallery(label='Output files (right-click to download)',
                              columns=2, height=200)
    gr.Markdown('---')
    gr.Markdown('**Tip:** For tighter VRAM (T4), drop `num_gaussians` to 65k and `steps` to 12. '
                'For higher quality on L4/A100, push to 262k and 30 steps. '
                '`.ply` opens in [Antimatter15](https://antimatter15.com/splat/), '
                '[gsplat.tech](https://gsplat.tech/), or LumaAI. '
                '`.splat` is the same data packed for web viewers.')

    q_btn.click(do_generate,
                [q_image, q_seed, q_steps, q_guidance, q_shift, q_num_g, q_random_seed],
                [q_html, q_info, q_files])

    # Welcome message
    def _welcome():
        return ('> **Welcome to TripoSplat** \u2014 image-to-3D Gaussian Splats in ~30s on L4.\
\
'
                '> **Quick Start:** upload an image \u2192 click **Generate 3D Gaussians** \u2192 '
                'download the `.ply` (open in [Antimatter15](https://antimatter15.com/splat/)) or `.splat` (web-viewer packed).\
\
'
                '> **For game-ready textured meshes:** use '
                '[Pixal3D](https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/Pixal3D_Colab.ipynb) '
                'instead \u2014 it produces PBR-textured GLB meshes from the same single-image input.')
    demo.load(_welcome, None, q_info)

clear_output()
demo.queue(max_size=4, default_concurrency_limit=2)
demo.launch(share=False, show_error=True, height=900, quiet=False)


In [ ]:
# @title STEP 5 — Keep-Alive (anti-disconnect)
# @markdown Prevents Colab from idling you out of the GPU. Runs a tiny JS console loop
# @markdown that pings the kernel every 30 minutes. Standard pattern across the suite.

import IPython
from google.colab import output

KEEP_ALIVE_JS = """
function KeepAlive() {
    setInterval(() => {
        console.log("keep-alive:", new Date().toISOString());
        google.colab.kernel.proxyProxy(0).then(() => {}).catch(() => {});
    }, 30 * 60 * 1000);
}
KeepAlive();
"""

display(IPython.display.Javascript(KEEP_ALIVE_JS))
print('[KeepAlive] Started — pings every 30 min. Run this in a separate cell from the Gradio UI.')


In [ ]:
# @title STEP 6 — Quick Test (single inference, no UI)
# @markdown Run a single end-to-end TripoSplat generation with parameterized settings.
# @markdown Saves `.ply` and `.splat` (the only useful outputs). For game-ready
# @markdown textured meshes use Pixal3D_Colab.ipynb instead.
import os, sys, time, pathlib
import torch
from PIL import Image
from IPython.display import FileLink, display  # noqa: F401
REPO_DIR = '/content/TripoSplat'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
# @markdown **Quick test parameters** (edit and run):
QUICK_STEPS = 12  # @param {type:"slider", min:5, max:50, step:1}
# @markdown *Flow-matching Euler steps. 10-20 recommended. More = slower + sharper.*
QUICK_NUM_GAUSSIANS = 131072  # @param {type:"slider", min:32768, max:262144, step:1024}
# @markdown *Total Gaussian count. Must be a multiple of 32. 131k default. T4-safe <= 65k.*
QUICK_SEED = 0  # @param {type:"integer"}
QUICK_GUIDANCE = 3.0  # @param {type:"slider", min:1.0, max:10.0, step:0.1}
# @markdown *Classifier-free guidance. ≤1 disables CFG. 3.0 is the model default.*
QUICK_SHIFT = 3.0  # @param {type:"slider", min:1.0, max:5.0, step:0.1}
# @markdown *Flow-matching time shift. 3.0 is the model default.*
QUICK_INPUT_IMAGE = ''  # @param {type:"string"}
# @markdown *Leave blank to auto-pick the first PNG/JPG under /content. Set to a specific path (e.g. /content/my_photo.jpg) to use that file. Drive paths work too (e.g. /content/drive/MyDrive/images/hero.png).*
QUICK_OUTPUT_DIR = '/content/triposplat_runs/quicktest'  # @param {type:"string"}
# Resolve input image
img_path = None
if QUICK_INPUT_IMAGE and QUICK_INPUT_IMAGE.strip():
    explicit = pathlib.Path(QUICK_INPUT_IMAGE.strip())
    if explicit.is_file():
        img_path = str(explicit)
    else:
        matches = list(pathlib.Path('/').glob(str(explicit).lstrip('/')))
        if matches:
            img_path = str(matches[0])
    if img_path is None:
        print(f'[QuickTest] WARNING: QUICK_INPUT_IMAGE={QUICK_INPUT_IMAGE!r} not found, falling back to auto-pick')
if img_path is None:
    INPUT_CANDIDATES = ['/content/upload.png', '/content/upload.jpg',
                        '/content/input.png', '/content/test.png']
    for p in INPUT_CANDIDATES:
        if os.path.exists(p):
            img_path = p
            break
if img_path is None:
    for p in pathlib.Path('/content').rglob('*.[pP][nN][gG]'):
        img_path = str(p); break
if img_path is None:
    for p in pathlib.Path('/content').rglob('*.[jJ][pP][gG]'):
        img_path = str(p); break
if img_path is None:
    raise SystemExit(
        'No input image found. Set QUICK_INPUT_IMAGE to a file path, '
        'or upload a PNG/JPG to /content/ (any subfolder works). Re-run this cell after.'
    )
print(f'[QuickTest] Using {img_path}')
img = Image.open(img_path).convert('RGBA')
print(f'[QuickTest] Image size: {img.size}')
print(f'[QuickTest] Params: steps={QUICK_STEPS} num_g={QUICK_NUM_GAUSSIANS} '
      f'guidance={QUICK_GUIDANCE} shift={QUICK_SHIFT}')
t0 = time.time()
gauss, prepared = PIPE.synthesize(
    img, seed=QUICK_SEED, steps=QUICK_STEPS,
    guidance_scale=QUICK_GUIDANCE, shift=QUICK_SHIFT,
    num_gaussians=QUICK_NUM_GAUSSIANS,
)
gen_dt = time.time() - t0
n_gauss = gauss.get_xyz.shape[0]
print(f'[QuickTest] Generation: {gen_dt:.1f}s — {n_gauss:,} Gaussians')
# Slugify the image filename
image_stem = pathlib.Path(img_path).stem
SLUG = ''.join(c if c.isalnum() else '_' for c in image_stem[:50]).strip('_') or 'quick'
if not SLUG or SLUG.startswith('.'):
    SLUG = 'quick'
print(f'[QuickTest] Using slug: {SLUG!r} (from {pathlib.Path(img_path).name})')
out_dir = pathlib.Path(QUICK_OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
# Save preprocessed image
prep_path = out_dir / f'{SLUG}_preprocessed.png'
prepared.save(str(prep_path))
print(f'  Preprocessed image: {prep_path}')
# Save .ply + .splat (the only useful outputs from TripoSplat)
print('[QuickTest] Saving native 3DGS formats ...')
paths = save_gaussians(gauss, str(out_dir), base_name=SLUG,
                       write_ply=True, write_splat=True)
# Download links
for fmt, p in paths.items():
    display(FileLink(str(p)))
    print(f'  {fmt.upper()}: {p} ({pathlib.Path(p).stat().st_size/1024:.0f} KB)')
if torch.cuda.is_available():
    vram = torch.cuda.memory_allocated() / 1024**3
    print(f'\n[QuickTest] Final VRAM: {vram:.1f} GB')
print(f'\n[QuickTest] DONE. All outputs in {QUICK_OUTPUT_DIR}')
print(f'[QuickTest] Open the .ply in [Antimatter15](https://antimatter15.com/splat/) to view in 3D.')


In [ ]:
# @title STEP 7 — Batch Generation from a folder
# @markdown Reads a folder of images (one per subject) and generates a 3DGS
# @markdown PLY + SPLAT for each. Outputs use the image filename as the slug.
# @markdown **For game-ready textured meshes use Pixal3D_Colab.ipynb instead.**
# @markdown
# @markdown ### Quality presets (general-use defaults)
# @markdown The defaults below are tuned for **balanced quality vs. speed**.
# @markdown For a 200+ image library on a T4 (15 GB), expect ~2-4 min per image.
# @markdown - **steps=30** (vs paper's 50): 2.5x sharper than the 12-step minimum,
# @markdown   1.7x slower than the 15-step old default. Best quality/time trade-off.
# @markdown - **num_gaussians=262144** (vs 131k old default): captures finer detail
# @markdown   (hair, fabric, edges). ~5 GB peak VRAM. T4-safe.
# @markdown - **guidance=3.0**: paper default. Higher = more adherence to input but
# @markdown   can double up fine details. Lower = more creative but less faithful.
# @markdown - **shift=3.0**: flow-matching time remap. Do not change.
# @markdown
# @markdown ### Speed presets (if you want faster)
# @markdown - For 200+ batch: steps=20, num_gaussians=131072 -> ~1 min/image, decent.
# @markdown - For 200+ batch: steps=12, num_gaussians=65536 -> ~30 s/image, preview.
# @markdown
# @markdown ### Crash safety
# @markdown - **Per-item Drive save**: each completed model is mirrored to Drive
# @markdown   immediately (not at the end). If Colab disconnects, completed items are safe.
# @markdown - **Resume support**: re-running the cell skips items that already have a
# @markdown   `.ply` in the output folder. Safe to re-run on the same folder.
# @markdown - **Colab T4 stability**: for batches > 20 images, consider running in
# @markdown   2-3 sessions (MAX_ITEMS + offset) to avoid the 90-min / 12-hr limits.

import os, sys, time, pathlib, shutil
import json as _json
import torch
from PIL import Image

REPO_DIR = '/content/TripoSplat'
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# @markdown **Batch parameters** (edit and run):
BATCH_INPUT_MODE = 'folder'  # @param ["folder", "txt list"]
# @markdown *Choose input source. 'folder' = scan a directory for PNGs/JPGs.
# @markdown 'txt list' = read paths from a text file (one per line, # comments OK).*
BATCH_INPUT_FOLDER = '/content/triposplat_runs/inputs'  # @param {type:"string"}
# @markdown *Folder to scan for images. Used when BATCH_INPUT_MODE='folder'.
# @markdown Defaults to /content/triposplat_runs/inputs (you must upload your images there
# @markdown or change this path). Drive paths work too.*
LIST_FILE = '/content/triposplat_runs/batch_list.txt'  # @param {type:"string"}
# @markdown *Text file with one image path per line. Used when BATCH_INPUT_MODE='txt list'. *
BATCH_RECURSIVE = False  # @param {type:"boolean"}
# @markdown *If true, scan subfolders too. The slug is prefixed with the parent folder name
# @markdown to avoid collisions across folders.*
BATCH_STEPS = 30  # @param {type:"slider", min:5, max:50, step:1}
# @markdown *Flow-matching denoising steps. 30 = quality default. 12 = preview. 50 = paper max.*
BATCH_NUM_GAUSSIANS = 262144  # @param {type:"slider", min:32768, max:262144, step:1024}
# @markdown *Total Gaussian budget. 262k = quality default. 131k = fast. 65k = T4-tight preview.*
BATCH_SEED = 0  # @param {type:"integer"}
BATCH_DO_RANDOM_SEED = False  # @param {type:"boolean"}
# @markdown *Override seed field with a random seed per item. Useful for diversity.*
BATCH_GUIDANCE = 3.0  # @param {type:"slider", min:1.0, max:10.0, step:0.1}
# @markdown *Classifier-free guidance. 3.0 = paper default. <2 loses input fidelity. >5 doubles details.*
BATCH_SHIFT = 3.0  # @param {type:"slider", min:1.0, max:5.0, step:0.1}
# @markdown *Flow-matching time shift. 3.0 = paper default. Do not change.*
OUT_ROOT = '/content/triposplat_runs/batch'  # @param {type:"string"}
# @markdown *Output folder. All .ply + .splat files go here flat under a timestamped subfolder.*
MAX_ITEMS = 0  # @param {type:"integer"}
# @markdown *Set > 0 to cap the batch (useful for testing on 3-5 images first).*
BATCH_DO_DRIVE_SAVE = True  # @param {type:"boolean"}
# @markdown *Mirror EACH completed item to Drive as it finishes (not at the end). Safe on disconnect.*
BATCH_RESUME = True  # @param {type:"boolean"}
# @markdown *Skip images that already have a .ply in the output folder. Safe to re-run.*
BATCH_VERBOSE = True  # @param {type:"boolean"}
# @markdown *Print per-item progress + ETA. Disable for quieter logs.*

# Resolve image list based on BATCH_INPUT_MODE
EXT_GLOBS = ('*.png', '*.PNG', '*.jpg', '*.JPG', '*.jpeg', '*.JPEG', '*.webp', '*.WEBP')
if BATCH_INPUT_MODE == 'folder':
    folder = pathlib.Path(BATCH_INPUT_FOLDER)
    if not folder.exists():
        folder.mkdir(parents=True, exist_ok=True)
        raise SystemExit(
            f'[Batch] Input folder {folder} did not exist (now created). '
            f'Upload your PNG/JPG images to that folder (or change BATCH_INPUT_FOLDER), '
            f'then re-run this cell.'
        )
    found = []
    for ext in EXT_GLOBS:
        if BATCH_RECURSIVE:
            found.extend(folder.rglob(ext))
        else:
            found.extend(folder.glob(ext))
    found = sorted(set(p for p in found if p.is_file()))
    lines = [str(p) for p in found]
    print(f'[Batch] Folder mode: {len(lines)} image(s) from {folder}'
          + (' (recursive)' if BATCH_RECURSIVE else ''))
else:  # 'txt list'
    list_path = pathlib.Path(LIST_FILE)
    if not list_path.exists():
        pngs = sorted(pathlib.Path('/content').rglob('*.png'))
        jpgs = sorted(pathlib.Path('/content').rglob('*.jpg')) + sorted(pathlib.Path('/content').rglob('*.jpeg'))
        examples = pngs + jpgs
        if not examples:
            raise SystemExit(f'No images found under /content. Set BATCH_INPUT_FOLDER to a folder with images, or put PNGs in /content/.')
        list_path.parent.mkdir(parents=True, exist_ok=True)
        list_path.write_text('\n'.join(str(p) for p in examples) + '\n')
        print(f'[Batch] No list file found; bootstrapped {len(examples)} images into {list_path}')
    lines = [l.strip() for l in list_path.read_text().splitlines()
             if l.strip() and not l.startswith('#')]
    print(f'[Batch] TXT list mode: {len(lines)} image(s) from {list_path}')
if not lines:
    raise SystemExit('[Batch] No images to process. Check BATCH_INPUT_FOLDER or LIST_FILE.')
if MAX_ITEMS:
    lines = lines[:MAX_ITEMS]
print(f'[Batch] {len(lines)} image(s) queued for processing.')

OUT_ROOT = pathlib.Path(OUT_ROOT)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
batch_dir = OUT_ROOT / f'batch_{int(time.time())}'
batch_dir.mkdir(parents=True, exist_ok=True)

# Persistent progress log: one JSON line per processed item.
# Survives Colab disconnects (we save it per-item, after Drive mirror).
PROGRESS_LOG = batch_dir / '_batch_progress.jsonl'
progress_records = []
if PROGRESS_LOG.exists():
    for line in PROGRESS_LOG.read_text().splitlines():
        if line.strip():
            try:
                progress_records.append(_json.loads(line))
            except Exception:
                pass
done_slugs = {r.get('slug') for r in progress_records if r.get('ok')}
if done_slugs and BATCH_VERBOSE:
    print(f'[Batch] Resume: {len(done_slugs)} already-completed item(s) in this folder; will skip.')

PIPE.load()
print(f'[Batch] Model loaded. Steps={BATCH_STEPS} num_gaussians={BATCH_NUM_GAUSSIANS} guidance={BATCH_GUIDANCE} shift={BATCH_SHIFT}')
if BATCH_VERBOSE:
    print(f'[Batch] Output folder: {batch_dir}')
    print(f'[Batch] Per-item Drive save: {BATCH_DO_DRIVE_SAVE}, Resume: {BATCH_RESUME}')
ok = 0
fail = 0
skipped = 0
t_start = time.time()
times = []

for i, line in enumerate(lines, 1):
    try:
        p = pathlib.Path(line)
        if not p.exists():
            print(f'  [{i:03d}] SKIP (not found): {line}')
            fail += 1
            continue
        # Slug from image filename. Include parent folder prefix to avoid
        # collisions across subfolders when in recursive mode.
        stem_slug = ''.join(c if c.isalnum() else '_' for c in p.stem[:50]).strip('_') or f'item_{i:02d}'
        if BATCH_INPUT_MODE == 'folder' and BATCH_RECURSIVE and p.parent != pathlib.Path(BATCH_INPUT_FOLDER):
            parent_slug = ''.join(c if c.isalnum() else '_' for c in p.parent.name[:20]).strip('_')
            SLUG = f'{parent_slug}_{stem_slug}' if parent_slug else stem_slug
        else:
            # Non-recursive or txt-list mode: just use the image stem
            SLUG = stem_slug
        SLUG = SLUG[:60].strip('_') or f'item_{i:02d}'

        # Resume: skip if this slug already has a .ply in the output folder
        out_ply = batch_dir / f'{SLUG}.ply'
        if BATCH_RESUME and out_ply.exists():
            if BATCH_VERBOSE:
                print(f'  [{i:03d}] SKIP (resume): {SLUG}.ply exists')
            skipped += 1
            continue

        img = Image.open(p).convert('RGBA')
        seed = int.from_bytes(os.urandom(2), 'big') if BATCH_DO_RANDOM_SEED else BATCH_SEED
        t0 = time.time()
        gauss, prepared = PIPE.synthesize(
            img, seed=seed, steps=BATCH_STEPS,
            guidance_scale=BATCH_GUIDANCE, shift=BATCH_SHIFT,
            num_gaussians=BATCH_NUM_GAUSSIANS,
        )
        gen_dt = time.time() - t0
        times.append(gen_dt)
        n_g = gauss.get_xyz.shape[0]
        out_paths = save_gaussians(gauss, str(batch_dir), base_name=SLUG,
                                    write_ply=True, write_splat=True)
        n_formats = len(out_paths)
        if BATCH_VERBOSE and times:
            avg_dt = sum(times) / len(times)
            eta = avg_dt * (len(lines) - i - skipped)
            print(f'  [{i:03d}] {p.name} -> {n_g:,} Gaussians + {n_formats} files ({gen_dt:.1f}s, ETA {eta/60:.1f} min)')
        else:
            print(f'  [{i:03d}] {p.name} -> {n_g:,} Gaussians + {n_formats} files ({gen_dt:.1f}s)')
        ok += 1

        # Per-item Drive mirror (was: at the end). This is the key
        # crash-safety fix - every completed item is safe immediately.
        if BATCH_DO_DRIVE_SAVE:
            drive_base = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/TripoSplat')
            if drive_base.parent.exists():
                try:
                    drive_dir = drive_base / batch_dir.name
                    drive_dir.mkdir(parents=True, exist_ok=True)
                    for src_p in out_paths.values():
                        shutil.copy2(str(src_p), str(drive_dir / pathlib.Path(src_p).name))
                except Exception as e:
                    print(f'  [{i:03d}] WARN: per-item Drive save failed: {e}')

        # Append to progress log AFTER successful save
        with open(PROGRESS_LOG, 'a') as f:
            f.write(_json.dumps({
                'i': i, 'slug': SLUG, 'src': str(p),
                'ok': True, 'n_gauss': n_g,
                'gen_dt': round(gen_dt, 2),
                'files': [pathlib.Path(v).name for v in out_paths.values()],
            }) + '\n')

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f'  [{i:03d}] EXCEPTION on {line}: {e}')
        import traceback
        traceback.print_exc()
        fail += 1
        # Log failure too so we do not retry on resume
        try:
            with open(PROGRESS_LOG, 'a') as f:
                f.write(_json.dumps({
                    'i': i, 'slug': SLUG if 'SLUG' in dir() else f'item_{i:02d}',
                    'src': line, 'ok': False, 'error': str(e)[:200],
                }) + '\n')
        except Exception:
            pass

elapsed = time.time() - t_start
print(f'\n[Batch] DONE: {ok} OK / {skipped} skipped (resume) / {fail} failed / {len(lines)} total in {elapsed/60:.1f} min -> {batch_dir}')
if times:
    print(f'[Batch] Per-item: avg {sum(times)/len(times):.1f}s, min {min(times):.1f}s, max {max(times):.1f}s')
if BATCH_DO_DRIVE_SAVE and ok > 0:
    drive_dir = pathlib.Path('/content/drive/MyDrive/AEI_3D_Out/TripoSplat') / batch_dir.name
    if drive_dir.exists():
        n_files = sum(1 for _ in drive_dir.rglob('*') if _.is_file() and _.name != '_batch_progress.jsonl')
        print(f'[Batch] Drive mirror: {n_files} file(s) in {drive_dir}')
if ok > 0:
    print(f'[Batch] Tip: zip the folder with `!cd {batch_dir} && zip -r {batch_dir.name}.zip .` to download as a single file.')
    print(f'[Batch] Next: run SplatTransform_Colab to compress .ply to .sog/.spz/GLB+KHR_gaussian_splatting for game engines.')
